# 23 — Existing LoRA + limits:0.8B512 and4B2,048

Like Notebook21: **A100 → Run all**, allow Drive, enter the current rate and balance.
Push this notebook, new scripts, and `results/lora-limit-sizes-inputs/` first.
Readable cells only; no encoded bundle or new training.

**0.8B LoRA-1 + thinking limit512:**464 questions once.
**4B LoRA-1 + thinking limit2,048:**the same464 questions once.
Both adapters are already saved in the project and checked before setup.
The completed2B LoRA + limit1,024 result remains **41.2% (191/464)** and is not rerun.

A LoRA is a trained add-on. A token is a small piece of text.
These new methods start with thinking ON, stop its notes at the chosen token limit,
then allow the remaining output room for code. They are not thinking OFF.
Overall caps:4,096 for HumanEval+ and8,192 for LiveCodeBench.
Seed3407; same sampling as earlier. Repeat stopping stays ON for both sizes.

**Question:** does adding the trained adapter help with the same free limit?
**Hypothesis:** more passing answers or comparable accuracy with fewer output tokens.
**Change:** add LoRA-1 to limit512 or limit2048.
**Measures:** passing tests, thinking/output tokens, and uncertainty from shared problems.
**Compare:** saved first attempts for the same limit, LoRA alone, OFF, and ON.
**Support:** a positive accuracy-gap interval against the same limit alone.
**Contradiction:** worse accuracy, no useful gain, or extra cost without benefit.
**Data:** the existing464 IDs. This is retrospective, not a fresh held-out exam.

Separate total estimated ceiling: **40 compute units** including setup and resumes.
This does not reset old notebooks' ledgers. It ends when done; the ceiling is not a spending target.
Save the final ZIP. If an early setup cell fails, disconnect manually.
No accuracy is filled into the thesis until actual saved results are checked.


In [ ]:
UNIT_CAP = 40.0
RESERVE_UNITS = 1.5
GENERATION_RESERVE_UNITS = 3.0
PREFLIGHT_PASSED = False
PROBLEMS_READY = False
print("0.8B LoRA + limit512, then4B LoRA + limit2048 •464 each • no training")


## 1. GPU, Drive, GitHub, and both adapters

Check exact saved weights and code revisions before installation.


In [ ]:
import os, sys, json, time, math, subprocess, hashlib, signal, shutil
from pathlib import Path
os.chdir("/content")
gpu_check = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
if gpu_check.returncode:
    raise RuntimeError("Choose A100 GPU in Runtime → Change runtime type.")
GPU, MEMORY = [x.strip() for x in gpu_check.stdout.strip().splitlines()[0].split(",")]
GPU_GB = float(MEMORY) / 1024
if "A100" not in GPU or GPU_GB < 35:
    raise RuntimeError(f"Expected A100, got {GPU} ({GPU_GB:.0f} GB).")
BATCH = 128 if GPU_GB >= 70 else 64
print(f"GPU: {GPU} ({GPU_GB:.0f} GB) • batch {BATCH}")
from google.colab import drive
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/stop-overthinking")
OUT = DRIVE / "results/lora-limit-sizes"
OUT.mkdir(parents=True, exist_ok=True)
probe = OUT / ".write-check"; probe.write_text("ok"); probe.unlink()

def resource_number(prompt):
    try:
        value = float(input(prompt).strip())
    except ValueError:
        raise RuntimeError("Enter the number shown by Colab, then rerun this cell.")
    if not math.isfinite(value) or value <= 0:
        raise RuntimeError("Enter a positive number shown by Colab.")
    return value
UNIT_RATE = resource_number("Compute units per hour (your screenshot showed 6.77): ")
AVAILABLE_NOW = resource_number("Available compute units right now (the balance, not the rate): ")
LEDGER = OUT / "compute-ledger-v2.json"
OLD_LEDGER = OUT / "compute-ledger.json"
if OLD_LEDGER.exists():
    print("Keeping the previous attempt ledger unchanged:", OLD_LEDGER)
if LEDGER.exists():
    budget = json.loads(LEDGER.read_text())
    # The user's fresh resource balance includes idle/disconnected usage.
    BASE_USED = max(float(budget.get("estimated_units", 0)), float(budget["initial_balance"]) - AVAILABLE_NOW, 0.0)
else:
    budget = {"version": 2, "initial_balance": AVAILABLE_NOW, "sessions": 0}
    BASE_USED = 0.0
SESSION_START = time.monotonic()
budget.update(estimated_units=BASE_USED, unit_rate=UNIT_RATE, unit_cap=UNIT_CAP,
              sessions=budget.get("sessions", 0) + 1, gpu=GPU, active=True,
              note="Notebook23 only. Estimate includes current-session time and confirmed balance changes on resume.")
def save_ledger(value):
    tmp = LEDGER.with_suffix(".tmp")
    with tmp.open("w") as f:
        json.dump(value, f, indent=2); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, LEDGER)
def tick():
    budget["estimated_units"] = BASE_USED + (time.monotonic() - SESSION_START) / 3600 * UNIT_RATE
    save_ledger(budget)
def remaining_seconds(reserve=RESERVE_UNITS):
    tick()
    return max(0.0, (min(UNIT_CAP - budget["estimated_units"], AVAILABLE_NOW - (budget["estimated_units"] - BASE_USED)) - reserve) / UNIT_RATE * 3600)
save_ledger(budget)
if remaining_seconds() < 60:
    raise RuntimeError("Not enough budget left for this run. Saved work is safe; review the ledger and Colab balance.")

REPO = "https://github.com/mahmudulhaquequdrati/stop-overthinking-thesis.git"
WORK = Path("/content/thesis")
if not (WORK / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "master", REPO, str(WORK)], check=True, timeout=300)
else:
    subprocess.run(["git", "-C", str(WORK), "pull", "--ff-only", "origin", "master"], check=True, timeout=300)
INPUTS = WORK / "results/lora-limit-sizes-inputs"
need = [WORK / "scripts/run_lora_limit_sizes.py", WORK / "scripts/prepare_lora_limit_data.py",
        WORK / "scripts/compare_lora_limit_sizes.py", INPUTS / "manifest.json"]
if any(not p.exists() for p in need):
    raise RuntimeError("GitHub is missing this revision's input files/scripts. Push the latest project changes, then reopen this notebook.")
manifest = json.loads((INPUTS / "manifest.json").read_text())
for name, wanted in manifest["files"].items():
    path = WORK / name if name.startswith("scripts/") else INPUTS / Path(name).name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != wanted:
        raise RuntimeError("GitHub code/input revision does not match its manifest: " + name)
ADAPTERS = {}
for size in ("0.8b","4b"):
    spec=manifest["models"][size]
    source=WORK/spec["adapter_path"]
    def sha_file(path):
        h=hashlib.sha256()
        with path.open("rb") as f:
            for part in iter(lambda:f.read(1024*1024),b""):h.update(part)
        return h.hexdigest()
    if sha_file(source/"adapter_model.safetensors")!=spec["adapter_sha256"] or sha_file(source/"adapter_config.json")!=spec["config_sha256"]:
        raise RuntimeError("Adapter mismatch: "+size)
    dest=Path("/content/lora-limit-23")/size
    dest.mkdir(parents=True,exist_ok=True)
    for name in ("adapter_model.safetensors","adapter_config.json"):shutil.copy2(source/name,dest/name)
    ADAPTERS[size]=dest
PROBLEMS = Path("/content/lora-limit-23/problems-464.json")
WHEELS = DRIVE / "wheels"; WHEELS.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(WORK / "scripts"))
os.chdir(WORK)
print("GitHub, Drive, LoRA weights, and input manifest checked.")
print(f"This notebook budget used: {budget['estimated_units']:.2f}/{UNIT_CAP:.0f} units.")


## 2. Monitored commands and installation

Use the successful Notebook21 package setup, with logs and budget stops.


In [ ]:
def run_process(command, tag, timeout=1200, reserve=RESERVE_UNITS):
    allowed = min(timeout, remaining_seconds(reserve))
    if allowed < 30:
        print(f"Budget stop before {tag}; saved work is safe.")
        return False
    log = OUT / (tag + ".log")
    print("Starting", tag, "• maximum", round(allowed / 60, 1), "minutes", flush=True)
    with log.open("a", buffering=1) as stream:
        position = log.stat().st_size
        proc = subprocess.Popen(command, cwd=WORK, stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
        started = time.monotonic()
        try:
            while proc.poll() is None:
                tick()
                if time.monotonic() - started >= allowed or remaining_seconds(reserve) <= 0:
                    os.killpg(proc.pid, signal.SIGTERM)
                    try:
                        proc.wait(timeout=5)
                    except subprocess.TimeoutExpired:
                        os.killpg(proc.pid, signal.SIGKILL); proc.wait()
                    print(f"Stopped {tag} at the time/budget limit. Log: {log}")
                    return False
                with log.open() as reader:
                    reader.seek(position); tail = reader.read(); position = reader.tell()
                if tail:
                    print(tail[-3000:], end="", flush=True)
                time.sleep(3)
        except BaseException:
            if proc.poll() is None:
                os.killpg(proc.pid, signal.SIGTERM)
                try:
                    proc.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    os.killpg(proc.pid, signal.SIGKILL); proc.wait()
            raise
        finally:
            tick()
    if proc.returncode != 0:
        tail = log.read_text(errors="replace")[-5000:]
        raise RuntimeError(f"{tag} failed. Saved work is safe. Last log lines:\n{tail}")
    return True

def pip_install(*packages, tag="install-packages", timeout=1200):
    command = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *packages]
    if not run_process(command, tag, timeout):
        raise RuntimeError("Setup stopped at the time/budget limit. Read " + str(OUT / (tag + ".log")))


In [ ]:
pip_install("transformers>=5.15,<6", "peft>=0.20,<0.22", "accelerate", "safetensors",
            "datasets", "flash-linear-attention", "ninja", "packaging", "wheel")
# TorchAO is optional and unused by this 16-bit, unquantized inference run.
# Colab's old copy causes PEFT adapter injection to fail before generation.
if not run_process([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], "remove-unused-torchao", 120):
    raise RuntimeError("Not enough time to remove the incompatible optional TorchAO backend.")
print("Package installation finished. Model download has not started.")


In [ ]:
import urllib.request

def torch_info():
    probe = "import json,sys,torch; print(json.dumps(dict(ver=torch.__version__.split('+')[0], cuda=(torch.version.cuda or '').split('.')[0], abi=str(torch.compiled_with_cxx11_abi()).upper(), py='cp%d%d'%sys.version_info[:2])))"
    checked = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True, timeout=90)
    if checked.returncode:
        raise RuntimeError("PyTorch check failed: " + checked.stderr[-1500:])
    info = json.loads(checked.stdout.strip().splitlines()[-1])
    info["tag"] = ".".join(info["ver"].split(".")[:2])
    if not info["cuda"]:
        raise RuntimeError("PyTorch has no CUDA. Choose a fresh A100 runtime.")
    return info

def fast_check():
    probe = "import causal_conv1d, fla; from transformers.models.qwen3_5.modeling_qwen3_5 import Qwen3_5ForCausalLM"
    return subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True, timeout=120)

def wheel_tail(info):
    return f"+cu{info['cuda']}torch{info['tag']}cxx11abi{info['abi']}-{info['py']}-{info['py']}-linux_x86_64.whl"

request = urllib.request.Request("https://api.github.com/repos/Dao-AILab/causal-conv1d/releases/tags/v1.7.0", headers={"User-Agent": "thesis-notebook-23"})
assets = None
def released_wheel(info):
    global assets
    if assets is None:
        with urllib.request.urlopen(request, timeout=30) as response:
            assets = json.load(response).get("assets", [])
    return next((a for a in assets if wheel_tail(info) in a["name"]), None)

def install_ready(info):
    saved = sorted(p for p in WHEELS.glob("causal_conv1d*.whl") if wheel_tail(info) in p.name)
    if saved:
        try:
            pip_install("--no-deps", str(saved[0]), tag="install-convolution")
            if fast_check().returncode == 0:
                return True
        except RuntimeError as error:
            print("Saved wheel did not load:", str(error)[-500:])
    asset = released_wheel(info)
    if not asset:
        return False
    dest = WHEELS / asset["name"]
    temporary = str(dest) + ".part"
    download = "import urllib.request,os; urllib.request.urlretrieve(" + repr(asset["browser_download_url"]) + ", " + repr(temporary) + "); os.replace(" + repr(temporary) + ", " + repr(str(dest)) + ")"
    if not run_process([sys.executable, "-c", download], "download-wheel", 300):
        raise RuntimeError("Wheel download exceeded the remaining setup time.")
    pip_install("--no-deps", str(dest), tag="install-convolution")
    return fast_check().returncode == 0

info = torch_info()
print("Current environment:", info)
check = fast_check()
if check.returncode != 0 and not install_ready(info):
    print("No compatible ready wheel for this PyTorch/CUDA. Selecting a published ready-wheel stack.")
    target = dict(ver="2.9.1", tag="2.9", cuda="12", abi="TRUE", py=info["py"])
    if not released_wheel(target):
        raise RuntimeError("No published wheel for this Python version. Stop here; no source compilation is attempted.")
    pip_install("torch==2.9.1", "torchvision==0.24.1", "torchaudio==2.9.1",
                "--index-url", "https://download.pytorch.org/whl/cu128", tag="compatible-pytorch")
    info = torch_info()
    print("Compatible environment:", info)
    if not install_ready(info):
        raise RuntimeError("The compatible wheel did not load. See saved logs; model generation has not started.")
check = fast_check()
if check.returncode != 0:
    raise RuntimeError("FAST PATH OFF: " + check.stderr[-2000:])
print("FAST PATH ON")
run_process([sys.executable, "-m", "pip", "freeze"], "package-versions", 60)
os.chdir(WORK)


## 3. Check both adapters on CPU

Check PEFT injection without downloading a base model. Real tensor mapping is checked during GPU loading.


In [ ]:
def worker(size,phase):
    folder=OUT/size;folder.mkdir(parents=True,exist_ok=True)
    return [sys.executable,str(WORK/"scripts/run_lora_limit_sizes.py"),"--model",size,
        "--phase",phase,"--problems",str(PROBLEMS),"--out",str(folder),
        "--adapter",str(ADAPTERS[size]),"--batch",str((64 if GPU_GB>=70 else 32) if size=="4b" else BATCH)]
for size in ("0.8b","4b"):
    if not run_process(worker(size,"preflight"),"preflight-"+size,180):
        raise RuntimeError("CPU compatibility check stopped: "+size)
PREFLIGHT_PASSED=True


## 4. Load and check the frozen464 questions

Reuse Notebook21’s verified Drive cache. No new questions or tests are selected.


In [ ]:
DATA_CACHE = DRIVE / "results/lora-limit-2b-inputs"
prepare_command = [sys.executable, str(WORK / "scripts/prepare_lora_limit_data.py"),
    "--manifest", str(INPUTS / "data-manifest.json"), "--templates", str(INPUTS / "problem-templates.json"),
    "--cache", str(DATA_CACHE), "--out", str(PROBLEMS)]
if not run_process(prepare_command, "prepare-data", 900):
    raise RuntimeError("Problem loading stopped; cached files and saved results are safe.")
problems = json.loads(PROBLEMS.read_text())["problems"]
assert len(problems) == len({p["task_id"] for p in problems}) == 464
assert set(p["task_id"] for p in problems) == set(manifest["problem_ids"])
assert not set(manifest["problem_ids"]).intersection(manifest["training_ids"])
assert all(p.get("test_plus") for p in problems if p["source"] == "humaneval")
assert all(0 < len(p["tests"]) <= 20 for p in problems if p["source"] == "lcb")
if not run_process([sys.executable, str(WORK / "scripts/grade_plus.py"), "--problems", str(PROBLEMS), "--source", "humaneval", "--check-official", "8"], "check-grader", 300):
    raise RuntimeError("Not enough time for the official-solution check.")
PROBLEMS_READY = True
print("464 problems checked. Official-solution grader check passed.")


## 5. Generate, grade, and compare each new method

0.8B runs first, then4B in a separate process. Saved answers are skipped on resume.
Each model’s first16 technical checks count toward464. There is no accuracy gate.
Each report compares the same shared IDs and saved first attempts.
Different historical environments and batches remain a limitation.


In [ ]:
from run_lora_limit import read_answers
for size in ("0.8b","4b"):
    if not globals().get("PREFLIGHT_PASSED",False) or not globals().get("PROBLEMS_READY",False):
        print("STOP: compatibility and data/grader checks must finish first.")
        break
    folder=OUT/size
    way="lora1-limit"+str(manifest["models"][size]["think_budget"])
    try:
        ok=run_process(worker(size,"generate"),"generate-"+size,24*3600,reserve=GENERATION_RESERVE_UNITS)
        if not ok:(folder/"generation-status.json").write_text(json.dumps({"state":"budget_stopped"}))
    except Exception as error:
        (folder/"generation-status.json").write_text(json.dumps({"state":"failed","error":str(error)[-5000:]}))
        print("GENERATION FAILED:",str(error)[-3000:])
    generated=sum(len(read_answers(folder/f"test-{way}-{tag}.jsonl",True)) for tag in ("he","lcb"))
    if generated:
        try:run_process(worker(size,"grade"),"grade-"+size,24*3600,reserve=RESERVE_UNITS)
        except Exception as error:print("GRADING FAILED:",str(error)[-3000:])
    else:print(size+": zero answers, skip grading. This is not0% accuracy.")
    report=[sys.executable,str(WORK/"scripts/compare_lora_limit_sizes.py"),"--model",size,
        "--out",str(folder),"--problems",str(PROBLEMS),"--baselines",str(INPUTS/f"baselines-{size}.json")]
    try:
        run_process(report,"compare-"+size,120,reserve=0.1)
        if (folder/"SUMMARY.md").exists():
            from IPython.display import display,Markdown
            display(Markdown((folder/"SUMMARY.md").read_text()))
    except Exception as error:print("Report unavailable:",str(error)[-2000:])
    if remaining_seconds(GENERATION_RESERVE_UNITS)<120:
        print("Budget stop; resume this same notebook. Saved answers stay on Drive.")
        break
print("Outputs:",OUT)


## 6. ZIP and release GPU

Send the ZIP back to update the full464 table. Both new scores remain unmeasured until then.


In [ ]:
tick()
budget["active"] = False
save_ledger(budget)
ZIP_BASE = DRIVE / "results/lora-limit-sizes-download"
try:
    shutil.make_archive(str(ZIP_BASE), "zip", root_dir=OUT)
    from google.colab import files
    print("Results ZIP:", str(ZIP_BASE) + ".zip")
    files.download(str(ZIP_BASE) + ".zip")
except Exception as error:
    print("Download was not started:", type(error).__name__, str(error)[:200])
    print("Your results are still saved on Drive:", OUT)
finally:
    print(f"Estimated charged time: {budget['estimated_units']:.2f}/{UNIT_CAP:.0f} units at the rate you entered.")
    print("Disconnecting this GPU runtime now. Reopen the same notebook to resume a partial run.")
    from google.colab import runtime
    runtime.unassign()
